In [0]:
%sql
  CREATE WIDGET TEXT catalog DEFAULT 'bootcamp';
  USE CATALOG IDENTIFIER(:catalog);

# Phase 5 Analytics (Q1–Q5)

In [0]:
-- Q1: What are monthly revenue, gross margin and units sold, by region and store?
SELECT
  d.year_month,
  s.region,
  s.store_name,
  SUM(f.net_amount)                                             AS revenue,
  SUM(f.margin_amount)                                          AS gross_margin,
  round(SUM(f.margin_amount) / nullif(SUM(f.net_amount), 0) * 100, 2) AS margin_pct,
  SUM(f.quantity)                                               AS units_sold
FROM gold_efkleia_urbanmart.fact_sales f
JOIN gold_efkleia_urbanmart.dim_date  d ON f.date_key  = d.date_key
JOIN gold_efkleia_urbanmart.dim_store s ON f.store_key = s.store_key
WHERE f.order_status = 'Completed'
GROUP BY d.year_month, s.region, s.store_name
ORDER BY d.year_month, s.region, s.store_name;

year_month,region,store_name,revenue,gross_margin,margin_pct,units_sold
2024-01,Attica,UrbanMart Athens 1,41367.93,12208.64,29.51,238
2024-01,Attica,UrbanMart Athens 8,34877.52,10206.38,29.26,218
2024-01,Attica,UrbanMart Online,32921.43,9316.94,28.30,193
2024-01,Attica,UrbanMart Piraeus 2,41820.46,12148.47,29.05,238
2024-01,Attica,UrbanMart Piraeus 9,34852.54,10365.90,29.74,201
2024-01,Central Macedonia,UrbanMart Thessaloniki 10,23732.05,7242.71,30.52,157
2024-01,Central Macedonia,UrbanMart Thessaloniki 3,25416.14,7843.14,30.86,146
2024-01,Crete,UrbanMart Heraklion 12,29841.04,8941.63,29.96,191
2024-01,Crete,UrbanMart Heraklion 5,36637.09,11210.29,30.60,211
2024-01,Epirus,UrbanMart Ioannina 7,27149.74,8237.57,30.34,162


In [0]:
-- Q2: Which product categories and brands drive the most margin?
SELECT
  p.category,
  p.brand,
  SUM(f.net_amount)                                             AS revenue,
  SUM(f.margin_amount)                                          AS gross_margin,
  round(SUM(f.margin_amount) / nullif(SUM(f.net_amount), 0) * 100, 2) AS margin_pct,
  round(SUM(f.margin_amount) * 100.0
        / SUM(SUM(f.margin_amount)) OVER (), 2)                 AS share_of_total_margin_pct,
  RANK() OVER (ORDER BY SUM(f.margin_amount) DESC)              AS margin_rank
FROM gold_efkleia_urbanmart.fact_sales f
JOIN gold_efkleia_urbanmart.dim_product p ON f.product_key = p.product_key
WHERE f.order_status = 'Completed'
GROUP BY p.category, p.brand
ORDER BY gross_margin DESC;

category,brand,revenue,gross_margin,margin_pct,share_of_total_margin_pct,margin_rank
Electronics,Nova,691435.68,229128.48,33.14,8.53,1
Grocery,Olympus,562102.06,220879.27,39.30,8.22,2
Sports,Olympus,652255.38,211084.74,32.36,7.85,3
Grocery,Nova,674323.45,199841.92,29.64,7.44,4
Home,Aegean,556413.46,175262.62,31.50,6.52,5
Home,Nova,484693.17,150761.93,31.10,5.61,6
Electronics,Aegean,588573.13,143041.81,24.30,5.32,7
Electronics,Helios,434485.97,122360.56,28.16,4.55,8
Apparel,Zephyr,412444.19,115695.63,28.05,4.31,9
Apparel,Helios,368175.80,110575.48,30.03,4.11,10


In [0]:
-- Q3: Who are our top customers per segment? (Top 5 by revenue, guests excluded)
SELECT segment, customer_rank, customer_id, full_name, city, revenue, orders, avg_order_value
FROM (
  SELECT
    c.segment,
    c.customer_id,
    c.full_name,
    c.city,
    SUM(f.net_amount)                                     AS revenue,
    COUNT(DISTINCT f.order_id)                            AS orders,
    round(SUM(f.net_amount) / COUNT(DISTINCT f.order_id), 2) AS avg_order_value,
    ROW_NUMBER() OVER (PARTITION BY c.segment
                       ORDER BY SUM(f.net_amount) DESC)   AS customer_rank
  FROM gold_efkleia_urbanmart.fact_sales f
  JOIN gold_efkleia_urbanmart.dim_customer c ON f.customer_key = c.customer_key
  WHERE f.order_status = 'Completed'
    AND f.customer_key <> -1
  GROUP BY c.segment, c.customer_id, c.full_name, c.city
)
WHERE customer_rank <= 5
ORDER BY segment, customer_rank;

segment,customer_rank,customer_id,full_name,city,revenue,orders,avg_order_value
Loyalty Gold,1,C0065,Giorgos Pappas,Piraeus,37217.82,24,1550.74
Loyalty Gold,2,C0345,Sofia Ioannou,Heraklion,33204.49,17,1953.21
Loyalty Gold,3,C0073,Nikos Georgiou,Piraeus,31291.02,25,1251.64
Loyalty Gold,4,C0225,Dimitris Ioannou,Athens,28942.73,18,1607.93
Loyalty Gold,5,C0126,Maria Vasileiou,Larissa,28246.13,22,1283.92
Loyalty Silver,1,C0043,Dimitris Oikonomou,Ioannina,32752.52,23,1424.02
Loyalty Silver,2,C0462,Nikos Vasileiou,Heraklion,29614.59,22,1346.12
Loyalty Silver,3,C0411,Sofia Georgiou,Athens,28718.72,19,1511.51
Loyalty Silver,4,C0104,Maria Ioannou,Patras,28651.28,19,1507.96
Loyalty Silver,5,C0422,Nikos Georgiou,Patras,27970.37,19,1472.12


In [0]:
-- Q3b (SCD2): Completed revenue by segment, as customers were WHEN they bought vs NOW.
-- Proves the point-in-time join; differences are small because only 24 customers changed.
SELECT 'At time of purchase' AS view, h.segment, SUM(f.net_amount) AS revenue
FROM gold_efkleia_urbanmart.fact_sales f
JOIN gold_efkleia_urbanmart.dim_customer_scd2 h ON f.customer_scd2_key = h.customer_scd2_key
WHERE f.order_status = 'Completed' AND f.customer_scd2_key <> -1
GROUP BY h.segment
UNION ALL
SELECT 'Current segment', cur.segment, SUM(f.net_amount)
FROM gold_efkleia_urbanmart.fact_sales f
JOIN gold_efkleia_urbanmart.dim_customer_scd2 h   ON f.customer_scd2_key = h.customer_scd2_key
JOIN gold_efkleia_urbanmart.dim_customer_scd2 cur ON cur.customer_id = h.customer_id AND cur.is_current
WHERE f.order_status = 'Completed' AND f.customer_scd2_key <> -1
GROUP BY cur.segment
ORDER BY segment, view;

view,segment,revenue
At time of purchase,Loyalty Gold,1902227.30
Current segment,Loyalty Gold,1908631.55
At time of purchase,Loyalty Silver,1839816.13
Current segment,Loyalty Silver,1844763.23
At time of purchase,Retail,5016417.41
Current segment,Retail,5005066.06


In [0]:
-- Q3c (SCD2): How did customers move between segments?
SELECT
  prev.segment                                         AS from_segment,
  cur.segment                                          AS to_segment,
  CASE
    WHEN prev.segment = cur.segment                   THEN 'City move only'
    WHEN cur.segment  = 'Retail'                      THEN 'Downgrade'
    WHEN prev.segment = 'Loyalty Gold'                THEN 'Downgrade'
    ELSE 'Upgrade'
  END                                                  AS movement,
  COUNT(*)                                             AS customers
FROM gold_efkleia_urbanmart.dim_customer_scd2 cur
JOIN gold_efkleia_urbanmart.dim_customer_scd2 prev
  ON  prev.customer_id = cur.customer_id
  AND prev.version_num = cur.version_num - 1
GROUP BY prev.segment, cur.segment
ORDER BY movement, customers DESC;

from_segment,to_segment,movement,customers
Retail,Retail,City move only,8
Loyalty Gold,Retail,Downgrade,4
Loyalty Silver,Retail,Downgrade,2
Retail,Loyalty Gold,Upgrade,5
Retail,Loyalty Silver,Upgrade,4
Loyalty Silver,Loyalty Gold,Upgrade,1


In [0]:
-- Q4: What is the return rate by category and store type?
SELECT
  p.category,
  s.store_type,
  COUNT_IF(f.order_status = 'Returned')                         AS returned_lines,
  COUNT(*)                                                      AS sold_lines,
  round(COUNT_IF(f.order_status = 'Returned') * 100.0 / COUNT(*), 2) AS return_rate_pct,
  round(SUM(CASE WHEN f.order_status = 'Returned' THEN f.net_amount ELSE 0 END) * 100.0
        / SUM(f.net_amount), 2)                                 AS return_rate_value_pct
FROM gold_efkleia_urbanmart.fact_sales f
JOIN gold_efkleia_urbanmart.dim_product p ON f.product_key = p.product_key
JOIN gold_efkleia_urbanmart.dim_store   s ON f.store_key   = s.store_key
WHERE f.order_status IN ('Completed', 'Returned')
GROUP BY p.category, s.store_type
ORDER BY return_rate_pct DESC;

category,store_type,returned_lines,sold_lines,return_rate_pct,return_rate_value_pct
Electronics,Mall,101,1397,7.23,6.71
Home,High Street,102,1444,7.06,6.28
Electronics,Online,26,369,7.05,6.45
Home,Mall,96,1408,6.82,6.49
Grocery,Mall,95,1407,6.75,7.08
Apparel,High Street,94,1441,6.52,7.25
Grocery,High Street,91,1476,6.17,5.71
Sports,High Street,89,1488,5.98,6.60
Sports,Mall,84,1427,5.89,6.31
Electronics,High Street,86,1475,5.83,6.49


In [0]:
-- Q5: How do weekdays vs weekends and online vs physical stores compare?
SELECT
  s.channel,
  CASE WHEN d.is_weekend THEN 'Weekend' ELSE 'Weekday' END     AS day_type,
  COUNT(DISTINCT f.order_id)                                   AS orders,
  SUM(f.net_amount)                                            AS revenue,
  round(SUM(f.net_amount) / COUNT(DISTINCT f.order_id), 2)     AS avg_order_value,
  COUNT(DISTINCT f.date_key)                                   AS trading_days,
  round(SUM(f.net_amount) / COUNT(DISTINCT f.date_key), 2)     AS avg_revenue_per_day,
  round(SUM(f.margin_amount) / nullif(SUM(f.net_amount), 0) * 100, 2) AS margin_pct
FROM gold_efkleia_urbanmart.fact_sales f
JOIN gold_efkleia_urbanmart.dim_date  d ON f.date_key  = d.date_key
JOIN gold_efkleia_urbanmart.dim_store s ON f.store_key = s.store_key
WHERE f.order_status = 'Completed'
GROUP BY s.channel, CASE WHEN d.is_weekend THEN 'Weekend' ELSE 'Weekday' END
ORDER BY s.channel, day_type;

channel,day_type,orders,revenue,avg_order_value,trading_days,avg_revenue_per_day,margin_pct
Online,Weekday,413,529490.05,1282.06,287,1844.91,28.70
Online,Weekend,172,216087.80,1256.32,121,1785.85,30.43
Physical,Weekday,4725,5999633.61,1269.76,522,11493.55,29.48
Physical,Weekend,1931,2393173.35,1239.34,208,11505.64,29.29
